# Exercises XP — Evaluating LLMs for Summarization

## Solution académique complète

Ce notebook met en place un workflow reproductible pour comparer plusieurs
modèles de résumé automatique :

- `google-t5/t5-small` ;
- `google-t5/t5-base` ;
- `openai-community/gpt2`.

Les modèles sont évalués avec :

- l’exact match accuracy ;
- une accuracy normalisée ;
- une accuracy personnalisée fondée sur un seuil de token-F1 ;
- ROUGE-1 ;
- ROUGE-2 ;
- ROUGE-L ;
- ROUGE-Lsum ;
- une inspection qualitative des sorties.

Le notebook fonctionne sans clé API. Une connexion internet est nécessaire
pour télécharger le dataset, les modèles et la métrique ROUGE.

## Ce que nous allons apprendre

À la fin du notebook, vous saurez :

1. charger des fichiers CSV ou un dataset Hugging Face avec pandas ;
2. nettoyer et échantillonner un corpus de résumé ;
3. générer des résumés par lots ;
4. utiliser T5 pour une tâche sequence-to-sequence ;
5. adapter GPT-2 à un prompt `TL;DR:` ;
6. calculer et interpréter plusieurs variantes de ROUGE ;
7. comprendre pourquoi l’exact match est inadapté au résumé libre ;
8. créer une métrique personnalisée ;
9. comparer les modèles globalement et ligne par ligne ;
10. diagnostiquer les sorties vides, répétitives ou trop longues.

## Workflow général

```text
Dataset
   ↓
Nettoyage et échantillonnage
   ↓
Articles + résumés de référence
   ↓
┌─────────────┬────────────┬──────────┐
│ T5-small    │ T5-base    │ GPT-2    │
└─────────────┴────────────┴──────────┘
   ↓
Résumés générés
   ↓
Exact match + métrique personnalisée + ROUGE
   ↓
Tableaux, graphiques et analyse qualitative
```

L’évaluation d’un résumé ne doit pas dépendre d’une seule métrique. Deux
résumés peuvent exprimer la même idée avec des formulations différentes.

## Part I — Installation et imports

In [ ]:
%pip install -q \
    "rouge_score==0.1.2" \
    "evaluate>=0.4,<0.5" \
    "accelerate>=1.0,<2.0" \
    "datasets>=3.0,<5.0" \
    "nltk>=3.8,<4.0" \
    "transformers>=4.45,<5.0" \
    "sentencepiece>=0.2,<1.0" \
    "pandas>=2.0,<3.0" \
    "matplotlib>=3.8,<4.0"

In [ ]:
import gc
import importlib.metadata as metadata
import os
import re
import string
import time
import warnings
from collections import Counter
from pathlib import Path
from time import perf_counter
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import evaluate
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    T5ForConditionalGeneration,
)

warnings.filterwarnings("ignore", category=FutureWarning)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Appareil :", DEVICE)
print("\nVersions principales :")

for package_name in [
    "torch",
    "transformers",
    "datasets",
    "evaluate",
    "rouge-score",
    "nltk",
    "pandas",
]:
    try:
        print(f"- {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: introuvable")

## Configuration de l’expérience

Les consignes demandent :

- 100 lignes d’entraînement ;
- 50 lignes de test.

Ces tailles sont respectées lors du chargement. Pour rendre l’exécution des
trois modèles raisonnable sur Google Colab, le mode rapide ne génère toutefois
les résumés que sur un sous-échantillon de l’entraînement.

Pour évaluer les 100 lignes :

```python
FAST_MODE = False
```

`t5-base` est nettement plus lourd que `t5-small`. Un GPU Colab est recommandé.

In [ ]:
TRAIN_SAMPLE_SIZE = 100
TEST_SAMPLE_SIZE = 50

FAST_MODE = True
FAST_GENERATION_SAMPLE_SIZE = 8

T5_SMALL_MODEL_ID = "google-t5/t5-small"
T5_BASE_MODEL_ID = "google-t5/t5-base"
GPT2_MODEL_ID = "openai-community/gpt2"

T5_MAX_INPUT_TOKENS = 512
T5_MAX_NEW_TOKENS = 64
GPT2_MAX_INPUT_TOKENS = 768
GPT2_MAX_NEW_TOKENS = 48

RANDOM_STATE = 42

# Renseignez ces chemins uniquement si vous avez téléversé les CSV.
TRAIN_CSV_PATH = ""
TEST_CSV_PATH = ""

print("Configuration prête.")

# Part II — Chargement, échantillonnage et exploration

## Format attendu

Les DataFrames doivent contenir :

- `prompt_text` : article à résumer ;
- `prompt_title` : résumé de référence.

Si les CSV ne sont pas fournis, le notebook télécharge
`abisee/cnn_dailymail`, puis renomme :

- `article` → `prompt_text` ;
- `highlights` → `prompt_title`.

Un petit corpus de secours est prévu si le téléchargement échoue.

In [ ]:
FALLBACK_ROWS = [
    {
        "prompt_text": (
            "The city opened a new public library on Monday. The building "
            "contains study rooms, computer labs, and a children's section. "
            "Officials expect it to serve thousands of residents each month."
        ),
        "prompt_title": "City opens new public library",
    },
    {
        "prompt_text": (
            "Scientists reported evidence of water ice near the moon's south "
            "pole. The finding may help future missions obtain water and fuel."
        ),
        "prompt_title": "Water ice found near moon's south pole",
    },
    {
        "prompt_text": (
            "The local football team won the championship after scoring in "
            "the final minute. Supporters celebrated throughout the night."
        ),
        "prompt_title": "Local team wins dramatic championship",
    },
    {
        "prompt_text": (
            "A technology company introduced a battery that charges in ten "
            "minutes and lasts longer than its previous model."
        ),
        "prompt_title": "Company unveils faster-charging battery",
    },
    {
        "prompt_text": (
            "Heavy rain caused flooding in several neighborhoods. Emergency "
            "workers moved dozens of families to temporary shelters."
        ),
        "prompt_title": "Flooding forces families into shelters",
    },
    {
        "prompt_text": (
            "The university launched a scholarship program for students from "
            "low-income households. The first awards will be issued next year."
        ),
        "prompt_title": "University launches scholarship program",
    },
    {
        "prompt_text": (
            "Doctors tested a new treatment that reduced recovery time in a "
            "small clinical study. Larger trials are still required."
        ),
        "prompt_title": "New treatment shows early promise",
    },
    {
        "prompt_text": (
            "The transport authority added electric buses to three routes to "
            "reduce fuel costs and urban air pollution."
        ),
        "prompt_title": "City adds electric buses",
    },
    {
        "prompt_text": (
            "A museum exhibition presents paintings and letters from several "
            "West African artists. It will remain open for four months."
        ),
        "prompt_title": "Museum opens West African art exhibition",
    },
    {
        "prompt_text": (
            "Farmers adopted a new irrigation method that uses less water "
            "while maintaining crop yields during the dry season."
        ),
        "prompt_title": "New irrigation method saves water",
    },
]

FALLBACK_DF = pd.DataFrame(FALLBACK_ROWS)


def validate_columns(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"prompt_text", "prompt_title"}
    missing = required_columns.difference(df.columns)

    if missing:
        raise ValueError(
            f"Colonnes manquantes : {sorted(missing)}. "
            "Le dataset doit contenir prompt_text et prompt_title."
        )

    return df[["prompt_text", "prompt_title"]].copy()


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = validate_columns(df)

    for column in ["prompt_text", "prompt_title"]:
        cleaned[column] = (
            cleaned[column]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    cleaned = cleaned[
        (cleaned["prompt_text"] != "")
        & (cleaned["prompt_title"] != "")
    ]

    cleaned = cleaned.drop_duplicates(
        subset=["prompt_text", "prompt_title"]
    )

    return cleaned.reset_index(drop=True)


def load_and_sample(
    csv_path: str,
    split_name: str,
    sample_size: int,
) -> pd.DataFrame:
    if csv_path:
        csv_file = Path(csv_path)

        if not csv_file.exists():
            raise FileNotFoundError(
                f"Le fichier n’existe pas : {csv_file}"
            )

        dataframe = pd.read_csv(csv_file)
        source_name = str(csv_file)

    else:
        try:
            pool_size = max(sample_size * 2, sample_size)

            dataset = load_dataset(
                "abisee/cnn_dailymail",
                "3.0.0",
                split=f"{split_name}[:{pool_size}]",
            )

            dataframe = (
                dataset.to_pandas()[["article", "highlights"]]
                .rename(
                    columns={
                        "article": "prompt_text",
                        "highlights": "prompt_title",
                    }
                )
            )

            source_name = "abisee/cnn_dailymail"

        except Exception as error:
            print(
                f"Chargement Hugging Face impossible pour {split_name}: "
                f"{error}"
            )
            print("Utilisation du corpus de secours.")
            dataframe = FALLBACK_DF.copy()
            source_name = "fallback local"

    dataframe = preprocess_dataframe(dataframe)

    if len(dataframe) > sample_size:
        dataframe = dataframe.sample(
            n=sample_size,
            random_state=RANDOM_STATE,
        )

    dataframe = dataframe.reset_index(drop=True)
    dataframe.attrs["source"] = source_name

    return dataframe


train_df = load_and_sample(
    TRAIN_CSV_PATH,
    "train",
    TRAIN_SAMPLE_SIZE,
)

test_df = load_and_sample(
    TEST_CSV_PATH,
    "test",
    TEST_SAMPLE_SIZE,
)

print("Source train :", train_df.attrs.get("source"))
print("Taille train :", len(train_df))
print("Taille test :", len(test_df))

## Exploration des données

In [ ]:
print("Première ligne de l’échantillon d’entraînement")
print("\nARTICLE")
print(train_df.loc[0, "prompt_text"][:1500])

print("\nRÉSUMÉ DE RÉFÉRENCE")
print(train_df.loc[0, "prompt_title"])

print("\nDataFrame train")
display(train_df.head())

print("\nDataFrame test")
display(test_df.head())

In [ ]:
def dataframe_statistics(df: pd.DataFrame, name: str) -> pd.DataFrame:
    article_words = df["prompt_text"].str.split().str.len()
    summary_words = df["prompt_title"].str.split().str.len()

    return pd.DataFrame(
        {
            "dataset": [name],
            "rows": [len(df)],
            "average_article_words": [article_words.mean()],
            "median_article_words": [article_words.median()],
            "average_summary_words": [summary_words.mean()],
            "median_summary_words": [summary_words.median()],
        }
    )


dataset_stats = pd.concat(
    [
        dataframe_statistics(train_df, "train"),
        dataframe_statistics(test_df, "test"),
    ],
    ignore_index=True,
)

display(dataset_stats.round(2))

## Corpus utilisé pour la génération

Les 100 lignes restent disponibles pour l’exploration. Le mode rapide sélectionne
un sous-ensemble déterministe pour les inférences, car charger successivement
T5-small, T5-base et GPT-2 est coûteux.

Cette distinction entre **taille du dataset** et **taille de l’évaluation
expérimentale** doit être clairement documentée dans un rapport.

In [ ]:
if FAST_MODE:
    generation_sample_size = min(
        FAST_GENERATION_SAMPLE_SIZE,
        len(train_df),
    )
else:
    generation_sample_size = len(train_df)

evaluation_df = (
    train_df.sample(
        n=generation_sample_size,
        random_state=RANDOM_STATE,
    )
    .reset_index(drop=True)
    .copy()
)

evaluation_df = evaluation_df.rename(
    columns={"prompt_title": "reference_summary"}
)

print("Lignes utilisées pour la génération :", len(evaluation_df))
display(evaluation_df.head())

# Part III — Résumé avec T5

T5 traite le résumé comme une tâche sequence-to-sequence.

Les entrées reçoivent le préfixe :

```text
summarize:
```

Paramètres importants :

- `max_input_tokens` : troncature des articles trop longs ;
- `max_new_tokens` : longueur maximale du résumé ;
- `num_beams` : nombre d’hypothèses conservées pendant la beam search ;
- `no_repeat_ngram_size` : réduction des répétitions.

In [ ]:
def batch_generator(
    items: Sequence[str],
    batch_size: int,
) -> Iterable[List[str]]:
    if batch_size <= 0:
        raise ValueError("batch_size doit être strictement positif.")

    for start_index in range(0, len(items), batch_size):
        yield list(items[start_index:start_index + batch_size])


def release_memory() -> None:
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def summarize_with_t5(
    texts: Sequence[str],
    model_name: str = T5_SMALL_MODEL_ID,
    batch_size: int = 4,
    max_input_tokens: int = T5_MAX_INPUT_TOKENS,
    max_new_tokens: int = T5_MAX_NEW_TOKENS,
    num_beams: int = 4,
) -> List[str]:
    if not texts:
        return []

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()

    summaries: List[str] = []

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "num_beams": num_beams,
        "no_repeat_ngram_size": 3,
        "length_penalty": 1.0,
        "do_sample": False,
    }

    if num_beams > 1:
        generation_kwargs["early_stopping"] = True

    try:
        for text_batch in batch_generator(texts, batch_size):
            prompted_batch = [
                f"summarize: {text}"
                for text in text_batch
            ]

            encoded = tokenizer(
                prompted_batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_input_tokens,
            ).to(DEVICE)

            with torch.inference_mode():
                generated_ids = model.generate(
                    **encoded,
                    **generation_kwargs,
                )

            decoded = tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            summaries.extend(summary.strip() for summary in decoded)

            del encoded, generated_ids
            release_memory()

    finally:
        del model, tokenizer
        release_memory()

    return summaries

## Génération avec T5-small

In [ ]:
model_runtimes: Dict[str, float] = {}

t5_small_start = perf_counter()

t5_small_summaries = summarize_with_t5(
    evaluation_df["prompt_text"].tolist(),
    model_name=T5_SMALL_MODEL_ID,
    batch_size=4 if torch.cuda.is_available() else 2,
    max_new_tokens=T5_MAX_NEW_TOKENS,
    num_beams=4,
)

model_runtimes["t5-small"] = perf_counter() - t5_small_start

evaluation_df["t5_small_summary"] = t5_small_summaries

print(
    f"T5-small terminé en "
    f"{model_runtimes['t5-small']:.2f} secondes."
)

display(
    evaluation_df[
        [
            "reference_summary",
            "t5_small_summary",
        ]
    ].head()
)

# Part IV — Accuracy

## Exact match accuracy

La métrique vaut 1 uniquement lorsque la prédiction est exactement identique à
la référence.

\[
Accuracy =
\frac{\text{nombre de correspondances exactes}}
     {\text{nombre total de résumés}}
\]

Pour du texte libre, cette métrique est beaucoup trop sévère. Une différence de
majuscule, de ponctuation ou une paraphrase correcte suffit pour produire zéro.

In [ ]:
def validate_prediction_reference_lengths(
    predictions: Sequence[str],
    references: Sequence[str],
) -> None:
    if len(predictions) != len(references):
        raise ValueError(
            "Le nombre de prédictions doit être égal au nombre "
            "de références."
        )


def compute_accuracy(
    predictions: Sequence[str],
    references: Sequence[str],
) -> float:
    validate_prediction_reference_lengths(
        predictions,
        references,
    )

    if not references:
        return 0.0

    matches = sum(
        prediction.strip() == reference.strip()
        for prediction, reference in zip(
            predictions,
            references,
        )
    )

    return matches / len(references)


exact_match_accuracy = compute_accuracy(
    evaluation_df["t5_small_summary"].tolist(),
    evaluation_df["reference_summary"].tolist(),
)

print(
    f"Exact-match accuracy de T5-small : "
    f"{exact_match_accuracy:.4f}"
)

## Accuracy normalisée et accuracy personnalisée

Nous ajoutons deux variantes pédagogiques.

### Normalized exact match

Avant la comparaison :

- conversion en minuscules ;
- suppression de la ponctuation ;
- suppression des articles anglais ;
- normalisation des espaces.

### Thresholded token-F1 accuracy

Une prédiction est considérée comme acceptable si son token-F1 avec la
référence dépasse un seuil.

Cette métrique est plus souple, mais elle reste lexicale et ne garantit ni la
factualité ni l’équivalence sémantique.

In [ ]:
def normalize_summary(text: str) -> str:
    normalized = text.lower()
    normalized = normalized.translate(
        str.maketrans("", "", string.punctuation)
    )
    normalized = re.sub(r"\b(a|an|the)\b", " ", normalized)
    normalized = " ".join(normalized.split())
    return normalized


def compute_normalized_accuracy(
    predictions: Sequence[str],
    references: Sequence[str],
) -> float:
    validate_prediction_reference_lengths(
        predictions,
        references,
    )

    if not references:
        return 0.0

    matches = sum(
        normalize_summary(prediction)
        == normalize_summary(reference)
        for prediction, reference in zip(
            predictions,
            references,
        )
    )

    return matches / len(references)


def token_f1(
    prediction: str,
    reference: str,
) -> float:
    prediction_tokens = normalize_summary(prediction).split()
    reference_tokens = normalize_summary(reference).split()

    if not prediction_tokens or not reference_tokens:
        return float(prediction_tokens == reference_tokens)

    prediction_counts = Counter(prediction_tokens)
    reference_counts = Counter(reference_tokens)

    common_tokens = sum(
        min(prediction_counts[token], reference_counts[token])
        for token in prediction_counts
    )

    if common_tokens == 0:
        return 0.0

    precision = common_tokens / len(prediction_tokens)
    recall = common_tokens / len(reference_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )


def compute_threshold_accuracy(
    predictions: Sequence[str],
    references: Sequence[str],
    minimum_token_f1: float = 0.5,
) -> float:
    validate_prediction_reference_lengths(
        predictions,
        references,
    )

    if not 0 <= minimum_token_f1 <= 1:
        raise ValueError(
            "minimum_token_f1 doit être compris entre 0 et 1."
        )

    if not references:
        return 0.0

    accepted = sum(
        token_f1(prediction, reference)
        >= minimum_token_f1
        for prediction, reference in zip(
            predictions,
            references,
        )
    )

    return accepted / len(references)


t5_small_normalized_accuracy = compute_normalized_accuracy(
    evaluation_df["t5_small_summary"].tolist(),
    evaluation_df["reference_summary"].tolist(),
)

t5_small_threshold_accuracy = compute_threshold_accuracy(
    evaluation_df["t5_small_summary"].tolist(),
    evaluation_df["reference_summary"].tolist(),
    minimum_token_f1=0.5,
)

accuracy_demo = pd.DataFrame(
    {
        "metric": [
            "Exact match",
            "Normalized exact match",
            "Token-F1 acceptance @ 0.50",
        ],
        "score": [
            exact_match_accuracy,
            t5_small_normalized_accuracy,
            t5_small_threshold_accuracy,
        ],
    }
)

display(accuracy_demo)

# Part V — Implémentation de ROUGE

ROUGE compare les chevauchements lexicaux entre une prédiction et une
référence.

- **ROUGE-1** : unigrammes ;
- **ROUGE-2** : bigrammes ;
- **ROUGE-L** : plus longue sous-séquence commune ;
- **ROUGE-Lsum** : variante adaptée aux résumés multi-phrases.

La segmentation en phrases et l’ajout de retours à la ligne sont utiles pour
ROUGE-Lsum.

In [ ]:
rouge_metric = evaluate.load("rouge")


def sentence_tokenize_safe(text: str) -> List[str]:
    cleaned_text = str(text or "").strip()

    if not cleaned_text:
        return []

    try:
        return sent_tokenize(cleaned_text)
    except LookupError:
        return [
            sentence.strip()
            for sentence in re.split(
                r"(?<=[.!?])\s+",
                cleaned_text,
            )
            if sentence.strip()
        ]


def preprocess_for_rouge(text: str) -> str:
    sentences = sentence_tokenize_safe(text)
    return "\n".join(sentences)


def compute_rouge_score(
    predictions: Sequence[str],
    references: Sequence[str],
    use_stemmer: bool = True,
) -> Dict[str, float]:
    validate_prediction_reference_lengths(
        predictions,
        references,
    )

    normalized_predictions = [
        preprocess_for_rouge(prediction)
        for prediction in predictions
    ]

    normalized_references = [
        preprocess_for_rouge(reference)
        for reference in references
    ]

    scores = rouge_metric.compute(
        predictions=normalized_predictions,
        references=normalized_references,
        use_stemmer=use_stemmer,
    )

    return {
        metric_name: float(metric_value)
        for metric_name, metric_value in scores.items()
    }


t5_small_rouge = compute_rouge_score(
    evaluation_df["t5_small_summary"].tolist(),
    evaluation_df["reference_summary"].tolist(),
)

display(
    pd.DataFrame(
        [t5_small_rouge],
        index=["t5-small"],
    ).round(4)
)

# Part VI — Comprendre le comportement de ROUGE

## 1. Prédiction identique

Tous les scores doivent être proches de 1.

## 2. Prédiction vide

Tous les scores doivent être nuls.

## 3. Stemming

Le stemming rapproche des formes morphologiques telles que `run`, `runs` et
`running`.

## 4. N-grammes

ROUGE-1 peut rester élevé lorsque plusieurs mots sont partagés, tandis que
ROUGE-2 baisse fortement si leur ordre local change.

In [ ]:
exact_reference = [
    "The local team won the championship."
]

exact_prediction = [
    "The local team won the championship."
]

empty_prediction = [""]

exact_match_rouge = compute_rouge_score(
    exact_prediction,
    exact_reference,
)

empty_rouge = compute_rouge_score(
    empty_prediction,
    exact_reference,
)

sanity_table = pd.DataFrame(
    [
        {"experiment": "Identical", **exact_match_rouge},
        {"experiment": "Empty prediction", **empty_rouge},
    ]
)

display(sanity_table.round(4))

In [ ]:
stemming_prediction = [
    "The athlete is running quickly."
]
stemming_reference = [
    "The athlete runs quickly."
]

stemming_table = pd.DataFrame(
    [
        {
            "setting": "Without stemming",
            **compute_rouge_score(
                stemming_prediction,
                stemming_reference,
                use_stemmer=False,
            ),
        },
        {
            "setting": "With stemming",
            **compute_rouge_score(
                stemming_prediction,
                stemming_reference,
                use_stemmer=True,
            ),
        },
    ]
)

display(stemming_table.round(4))

In [ ]:
ngram_reference = [
    "The new public library opened on Monday."
]

ngram_predictions = {
    "Exact": "The new public library opened on Monday.",
    "Partial ordered overlap": "The public library opened Monday.",
    "Same words, altered order": "Monday opened the library public new.",
    "Unrelated": "Scientists studied distant galaxies.",
}

ngram_rows = []

for experiment_name, prediction in ngram_predictions.items():
    scores = compute_rouge_score(
        [prediction],
        ngram_reference,
    )

    ngram_rows.append(
        {
            "experiment": experiment_name,
            **scores,
        }
    )

ngram_table = pd.DataFrame(ngram_rows)
display(ngram_table.round(4))

## Symétrie : une nuance importante

La version agrégée de ROUGE généralement renvoyée par `evaluate` utilise le
F1-score. Le F1 est symétrique lorsque prédiction et référence sont échangées.

Cependant, la **précision** et le **rappel** individuels ne sont pas
symétriques : ils s’échangent.

In [ ]:
detailed_scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

symmetry_reference = "the cat sat on the mat"
symmetry_prediction = "cat sat"

forward_scores = detailed_scorer.score(
    symmetry_reference,
    symmetry_prediction,
)

reverse_scores = detailed_scorer.score(
    symmetry_prediction,
    symmetry_reference,
)

symmetry_rows = []

for metric_name in ["rouge1", "rouge2", "rougeL"]:
    symmetry_rows.append(
        {
            "metric": metric_name,
            "direction": "reference → prediction",
            "precision": forward_scores[metric_name].precision,
            "recall": forward_scores[metric_name].recall,
            "f1": forward_scores[metric_name].fmeasure,
        }
    )
    symmetry_rows.append(
        {
            "metric": metric_name,
            "direction": "prediction → reference",
            "precision": reverse_scores[metric_name].precision,
            "recall": reverse_scores[metric_name].recall,
            "f1": reverse_scores[metric_name].fmeasure,
        }
    )

display(pd.DataFrame(symmetry_rows).round(4))

# Part VII — Comparaison de T5-small, T5-base et GPT-2

## Pourquoi ces modèles sont différents

### T5-small et T5-base

Ils sont conçus comme modèles sequence-to-sequence et comprennent le préfixe
`summarize:`.

### GPT-2

GPT-2 est un modèle causal généraliste, non spécialement entraîné pour résumer.
Nous utilisons un prompt de type :

```text
Article...

TL;DR:
```

Cette comparaison évalue donc à la fois la taille et l’adéquation de
l’architecture à la tâche. Un modèle plus grand n’est pas automatiquement mieux
adapté.

In [ ]:
def summarize_with_gpt2(
    texts: Sequence[str],
    model_name: str = GPT2_MODEL_ID,
    batch_size: int = 2,
    max_input_tokens: int = GPT2_MAX_INPUT_TOKENS,
    max_new_tokens: int = GPT2_MAX_NEW_TOKENS,
) -> List[str]:
    if not texts:
        return []

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "right"

    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)
    model.eval()

    safe_max_input_tokens = min(
        max_input_tokens,
        tokenizer.model_max_length - max_new_tokens - 1,
    )

    summaries: List[str] = []

    try:
        for text_batch in batch_generator(texts, batch_size):
            prompts = [
                f"{text}\n\nTL;DR:"
                for text in text_batch
            ]

            encoded = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=safe_max_input_tokens,
            ).to(DEVICE)

            padded_input_length = encoded["input_ids"].shape[1]

            with torch.inference_mode():
                generated_ids = model.generate(
                    **encoded,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    num_beams=1,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            continuation_ids = generated_ids[
                :,
                padded_input_length:,
            ]

            decoded = tokenizer.batch_decode(
                continuation_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            summaries.extend(
                summary.strip().split("\n")[0].strip()
                for summary in decoded
            )

            del encoded, generated_ids, continuation_ids
            release_memory()

    finally:
        del model, tokenizer
        release_memory()

    return summaries

## Génération avec T5-base

In [ ]:
t5_base_start = perf_counter()

t5_base_summaries = summarize_with_t5(
    evaluation_df["prompt_text"].tolist(),
    model_name=T5_BASE_MODEL_ID,
    batch_size=2 if torch.cuda.is_available() else 1,
    max_new_tokens=T5_MAX_NEW_TOKENS,
    num_beams=4,
)

model_runtimes["t5-base"] = perf_counter() - t5_base_start
evaluation_df["t5_base_summary"] = t5_base_summaries

print(
    f"T5-base terminé en "
    f"{model_runtimes['t5-base']:.2f} secondes."
)

## Génération avec GPT-2

In [ ]:
gpt2_start = perf_counter()

gpt2_summaries = summarize_with_gpt2(
    evaluation_df["prompt_text"].tolist(),
    model_name=GPT2_MODEL_ID,
    batch_size=2 if torch.cuda.is_available() else 1,
    max_new_tokens=GPT2_MAX_NEW_TOKENS,
)

model_runtimes["gpt2"] = perf_counter() - gpt2_start
evaluation_df["gpt2_summary"] = gpt2_summaries

print(
    f"GPT-2 terminé en "
    f"{model_runtimes['gpt2']:.2f} secondes."
)

## ROUGE par ligne

Le score global peut masquer des réussites et des échecs individuels. La
fonction suivante ajoute les F1 ROUGE de chaque résumé au DataFrame.

In [ ]:
def compute_rouge_per_row(
    dataframe: pd.DataFrame,
    prediction_column: str,
    reference_column: str = "reference_summary",
    use_stemmer: bool = True,
) -> pd.DataFrame:
    if prediction_column not in dataframe.columns:
        raise KeyError(
            f"Colonne absente : {prediction_column}"
        )

    if reference_column not in dataframe.columns:
        raise KeyError(
            f"Colonne absente : {reference_column}"
        )

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=use_stemmer,
    )

    output = dataframe.copy()

    rouge1_scores = []
    rouge2_scores = []
    rouge_l_scores = []

    for prediction, reference in zip(
        output[prediction_column],
        output[reference_column],
    ):
        scores = scorer.score(
            str(reference),
            str(prediction),
        )

        rouge1_scores.append(
            scores["rouge1"].fmeasure
        )
        rouge2_scores.append(
            scores["rouge2"].fmeasure
        )
        rouge_l_scores.append(
            scores["rougeL"].fmeasure
        )

    prefix = prediction_column.replace("_summary", "")

    output[f"{prefix}_rouge1"] = rouge1_scores
    output[f"{prefix}_rouge2"] = rouge2_scores
    output[f"{prefix}_rougeL"] = rouge_l_scores

    return output


per_row_df = evaluation_df.copy()

for prediction_column in [
    "t5_small_summary",
    "t5_base_summary",
    "gpt2_summary",
]:
    per_row_df = compute_rouge_per_row(
        per_row_df,
        prediction_column,
    )

display(
    per_row_df[
        [
            "reference_summary",
            "t5_small_summary",
            "t5_small_rougeL",
            "t5_base_summary",
            "t5_base_rougeL",
            "gpt2_summary",
            "gpt2_rougeL",
        ]
    ]
)

# Part VIII — Comparaison de tous les modèles

## Fonctions d’agrégation

In [ ]:
MODEL_COLUMNS = {
    "t5-small": "t5_small_summary",
    "t5-base": "t5_base_summary",
    "gpt2": "gpt2_summary",
}


def compare_models(
    rouge_results: Dict[str, Dict[str, float]],
    runtimes: Optional[Dict[str, float]] = None,
) -> pd.DataFrame:
    rows = []

    for model_name, model_scores in rouge_results.items():
        row = {
            "model": model_name,
            **model_scores,
        }

        if runtimes is not None:
            row["runtime_seconds"] = runtimes.get(
                model_name,
                np.nan,
            )

        rows.append(row)

    comparison = pd.DataFrame(rows)

    preferred_columns = [
        "model",
        "rouge1",
        "rouge2",
        "rougeL",
        "rougeLsum",
        "runtime_seconds",
    ]

    available_columns = [
        column
        for column in preferred_columns
        if column in comparison.columns
    ]

    return (
        comparison[available_columns]
        .sort_values("rougeL", ascending=False)
        .reset_index(drop=True)
    )


def compare_models_summaries(
    dataframe: pd.DataFrame,
    prediction_columns: Sequence[str],
    number_of_rows: int = 5,
) -> pd.DataFrame:
    required_columns = [
        "prompt_text",
        "reference_summary",
        *prediction_columns,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Colonnes manquantes : {missing_columns}"
        )

    comparison = dataframe[
        required_columns
    ].head(number_of_rows).copy()

    comparison["prompt_text"] = comparison[
        "prompt_text"
    ].apply(
        lambda text: textwrap.shorten(
            str(text),
            width=400,
            placeholder=" ...",
        )
    )

    return comparison

In [ ]:
rouge_results = {}

for model_name, prediction_column in MODEL_COLUMNS.items():
    rouge_results[model_name] = compute_rouge_score(
        evaluation_df[prediction_column].tolist(),
        evaluation_df["reference_summary"].tolist(),
    )

model_comparison_df = compare_models(
    rouge_results,
    runtimes=model_runtimes,
)

display(model_comparison_df.round(4))

## Résumés côte à côte

In [ ]:
side_by_side_df = compare_models_summaries(
    evaluation_df,
    prediction_columns=list(MODEL_COLUMNS.values()),
    number_of_rows=min(5, len(evaluation_df)),
)

display(side_by_side_df)

## Visualisation des scores ROUGE

In [ ]:
model_comparison_df.set_index("model")[
    ["rouge1", "rouge2", "rougeL", "rougeLsum"]
].plot(
    kind="bar",
    figsize=(10, 5),
)

plt.title("Comparaison des scores ROUGE")
plt.xlabel("Modèle")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Comparaison des métriques de type accuracy

In [ ]:
accuracy_rows = []

for model_name, prediction_column in MODEL_COLUMNS.items():
    predictions = evaluation_df[
        prediction_column
    ].tolist()

    references = evaluation_df[
        "reference_summary"
    ].tolist()

    accuracy_rows.append(
        {
            "model": model_name,
            "exact_match": compute_accuracy(
                predictions,
                references,
            ),
            "normalized_exact_match": (
                compute_normalized_accuracy(
                    predictions,
                    references,
                )
            ),
            "token_f1_acceptance_0_50": (
                compute_threshold_accuracy(
                    predictions,
                    references,
                    minimum_token_f1=0.5,
                )
            ),
        }
    )

accuracy_comparison_df = pd.DataFrame(
    accuracy_rows
)

display(accuracy_comparison_df.round(4))

In [ ]:
accuracy_comparison_df.set_index("model")[
    [
        "exact_match",
        "normalized_exact_match",
        "token_f1_acceptance_0_50",
    ]
].plot(
    kind="bar",
    figsize=(10, 5),
)

plt.title("Accuracy stricte et variantes personnalisées")
plt.xlabel("Modèle")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Diagnostic des sorties

Une bonne analyse ne se limite pas à ROUGE. Nous vérifions aussi :

- la proportion de sorties vides ;
- la longueur moyenne ;
- les sorties identiques à l’article ;
- la répétition excessive ;
- le temps d’inférence.

In [ ]:
def has_excessive_repetition(text: str) -> bool:
    tokens = normalize_summary(text).split()

    if len(tokens) < 6:
        return False

    bigrams = list(zip(tokens, tokens[1:]))

    if not bigrams:
        return False

    unique_ratio = len(set(bigrams)) / len(bigrams)
    return unique_ratio < 0.55


def model_output_diagnostics(
    dataframe: pd.DataFrame,
    model_columns: Dict[str, str],
) -> pd.DataFrame:
    rows = []

    for model_name, prediction_column in model_columns.items():
        predictions = dataframe[prediction_column].fillna("").astype(str)
        articles = dataframe["prompt_text"].fillna("").astype(str)

        rows.append(
            {
                "model": model_name,
                "empty_output_rate": (
                    predictions.str.strip().eq("").mean()
                ),
                "average_output_words": (
                    predictions.str.split().str.len().mean()
                ),
                "median_output_words": (
                    predictions.str.split().str.len().median()
                ),
                "exact_article_copy_rate": np.mean(
                    [
                        normalize_summary(prediction)
                        == normalize_summary(article)
                        for prediction, article in zip(
                            predictions,
                            articles,
                        )
                    ]
                ),
                "repetition_rate": np.mean(
                    [
                        has_excessive_repetition(prediction)
                        for prediction in predictions
                    ]
                ),
                "runtime_seconds": model_runtimes.get(
                    model_name,
                    np.nan,
                ),
            }
        )

    return pd.DataFrame(rows)


diagnostics_df = model_output_diagnostics(
    evaluation_df,
    MODEL_COLUMNS,
)

display(diagnostics_df.round(4))

# Expérience de personnalisation des paramètres

Nous comparons `num_beams=1` et `num_beams=4` sur quelques articles.

- `num_beams=1` correspond à une génération gloutonne ;
- `num_beams=4` explore plusieurs séquences candidates ;
- plus de beams peut améliorer la qualité, mais augmente le temps de calcul.

Les résultats dépendent du dataset et ne doivent pas être généralisés à partir
de quelques lignes.

In [ ]:
parameter_experiment_size = min(
    3,
    len(evaluation_df),
)

parameter_articles = evaluation_df[
    "prompt_text"
].head(parameter_experiment_size).tolist()

parameter_references = evaluation_df[
    "reference_summary"
].head(parameter_experiment_size).tolist()

beam_1_start = perf_counter()

beam_1_summaries = summarize_with_t5(
    parameter_articles,
    model_name=T5_SMALL_MODEL_ID,
    batch_size=parameter_experiment_size,
    max_new_tokens=T5_MAX_NEW_TOKENS,
    num_beams=1,
)

beam_1_runtime = perf_counter() - beam_1_start

beam_4_summaries = evaluation_df[
    "t5_small_summary"
].head(parameter_experiment_size).tolist()

beam_comparison = pd.DataFrame(
    [
        {
            "setting": "num_beams=1",
            **compute_rouge_score(
                beam_1_summaries,
                parameter_references,
            ),
            "runtime_seconds": beam_1_runtime,
        },
        {
            "setting": "num_beams=4",
            **compute_rouge_score(
                beam_4_summaries,
                parameter_references,
            ),
            "runtime_seconds": np.nan,
        },
    ]
)

display(beam_comparison.round(4))

In [ ]:
beam_summary_comparison = pd.DataFrame(
    {
        "reference": parameter_references,
        "beam_1": beam_1_summaries,
        "beam_4": beam_4_summaries,
    }
)

display(beam_summary_comparison)

# Rapport analytique automatique

In [ ]:
best_model_row = model_comparison_df.iloc[0]
best_model_name = best_model_row["model"]
best_rouge_l = best_model_row["rougeL"]

fastest_model_name = min(
    model_runtimes,
    key=model_runtimes.get,
)

print("SYNTHÈSE DES RÉSULTATS")
print("=" * 80)
print(
    f"Meilleur ROUGE-L sur cet échantillon : "
    f"{best_model_name} ({best_rouge_l:.4f})"
)
print(
    f"Modèle le plus rapide dans cette exécution : "
    f"{fastest_model_name} "
    f"({model_runtimes[fastest_model_name]:.2f} s)"
)
print(
    "\nL’exact match est généralement faible, car plusieurs "
    "résumés différents peuvent être valides."
)
print(
    "ROUGE mesure le chevauchement lexical, mais ne vérifie "
    "pas directement la factualité."
)
print(
    "La comparaison T5/GPT-2 mélange l’effet de la taille et "
    "l’effet de l’adaptation à la tâche de résumé."
)
print(
    "Une conclusion fiable nécessiterait plus d’exemples, "
    "plusieurs références et une évaluation humaine."
)

# Interprétation académique

## Pourquoi l’accuracy est-elle proche de zéro ?

Le résumé est une tâche ouverte. Une référence possible est :

> Local team wins championship.

Une prédiction également correcte peut être :

> Dramatic late goal gives local team the title.

Le sens est proche, mais l’exact match vaut zéro.

## Que mesure ROUGE ?

ROUGE favorise les prédictions qui réutilisent les mots ou séquences de la
référence. Il est utile pour comparer des systèmes sur un même corpus, mais il
ne mesure pas parfaitement :

- la factualité ;
- la cohérence ;
- la lisibilité ;
- l’importance des informations sélectionnées ;
- la qualité d’une paraphrase.

## Effet de la taille du modèle

T5-base possède davantage de paramètres que T5-small et peut produire des
représentations plus riches. Cependant, un gain n’est pas garanti :

- le dataset peut être petit ;
- les paramètres de génération peuvent être inadaptés ;
- les articles sont tronqués ;
- la référence peut favoriser certaines formulations.

## Pourquoi GPT-2 peut-il être moins performant ?

GPT-2 est un modèle causal généraliste. Il n’a pas été conçu comme modèle
sequence-to-sequence de résumé. Le prompt `TL;DR:` constitue une adaptation
simple, mais pas un fine-tuning.

La comparaison montre donc qu’une architecture adaptée à la tâche peut compter
autant que la taille brute du modèle.

# Méthode de débogage

Lorsqu’un résultat est mauvais, vérifiez les étapes dans cet ordre.

## 1. Données

- Les colonnes sont-elles correctes ?
- Les références sont-elles réellement des résumés ?
- Les cellules vides ont-elles été supprimées ?
- Les articles et références sont-ils bien alignés ?

## 2. Prétraitement

- Les espaces et retours à la ligne ont-ils été normalisés ?
- L’article a-t-il été trop fortement tronqué ?
- La ponctuation utile a-t-elle été supprimée ?

## 3. Génération

- Le préfixe `summarize:` est-il présent pour T5 ?
- Le prompt `TL;DR:` est-il présent pour GPT-2 ?
- `max_new_tokens` est-il suffisant ?
- Les sorties contiennent-elles le prompt d’entrée ?
- Le modèle et les tenseurs sont-ils sur le même appareil ?

## 4. Métriques

- Les listes ont-elles la même longueur ?
- Les prédictions et références sont-elles dans le bon ordre ?
- Le stemming est-il activé ?
- Les phrases sont-elles séparées par des retours à la ligne pour ROUGE-Lsum ?

## 5. Analyse

- Une mauvaise note provient-elle réellement d’un mauvais résumé ?
- La référence est-elle l’unique formulation acceptable ?
- Le modèle invente-t-il des faits malgré un ROUGE raisonnable ?

# Limites et extensions possibles

## Limites

1. Le mode rapide utilise peu d’articles.
2. CNN/DailyMail ne représente pas tous les domaines.
3. Les articles sont tronqués pour respecter les limites des modèles.
4. ROUGE est principalement lexical.
5. GPT-2 n’est pas fine-tuné pour le résumé.
6. Les temps dépendent fortement du matériel Colab.
7. Une seule référence est disponible par article.

## Extensions

- ajouter BERTScore pour la similarité sémantique ;
- ajouter BLEURT ou COMET-like evaluators adaptés ;
- vérifier la factualité avec un modèle de QA ou NLI ;
- faire évaluer les sorties par plusieurs humains ;
- comparer des modèles spécialisés comme BART ou PEGASUS ;
- tester plusieurs longueurs, beams et pénalités ;
- calculer des intervalles de confiance par bootstrap ;
- analyser séparément les articles courts et longs.

# Conclusion

Ce notebook a construit une procédure complète d’évaluation :

```text
Chargement → Nettoyage → Génération → Accuracy → ROUGE
                                      ↓
                         Comparaison par ligne
                                      ↓
                         Tableaux et graphiques
                                      ↓
                         Diagnostic qualitatif
```

Les principaux enseignements sont :

- l’exact match est trop sévère pour le résumé ;
- une métrique personnalisée peut être plus souple, mais doit être justifiée ;
- ROUGE-1, ROUGE-2 et ROUGE-L capturent des aspects différents ;
- le stemming modifie les chevauchements lexicaux ;
- le F1 ROUGE peut être symétrique, contrairement à sa précision et son rappel ;
- la taille du modèle n’est qu’un facteur parmi d’autres ;
- l’adaptation du modèle à la tâche est essentielle ;
- une évaluation fiable combine métriques automatiques et analyse humaine.

# Références techniques

- Hugging Face Transformers — Summarization:
  https://huggingface.co/docs/transformers/tasks/summarization
- Hugging Face Evaluate:
  https://huggingface.co/docs/evaluate/
- Hugging Face Datasets — Loading:
  https://huggingface.co/docs/datasets/loading
- CNN/DailyMail:
  https://huggingface.co/datasets/abisee/cnn_dailymail
- T5-small:
  https://huggingface.co/google-t5/t5-small
- T5-base:
  https://huggingface.co/google-t5/t5-base
- GPT-2:
  https://huggingface.co/openai-community/gpt2
- NLTK Tokenization:
  https://www.nltk.org/api/nltk.tokenize.html